# Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline 
# pour afficher facilement les graphiques 
from urllib.request import urlopen
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import os
import re
from pathlib import Path
from nltk.stem import SnowballStemmer
import spacy


# Variables 

In [ ]:
stemmer = SnowballStemmer("french")
nlp = spacy.load("fr_core_news_sm")

In [11]:
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd().parent

BULLETINS = BASE_DIR / "BULLETINS"
DATA = BASE_DIR / "data"
OUTPUT = BASE_DIR / "output"

# 0. Découpage du corpus 

In [ ]:
def decoupage(corpus, balise):
    try : 
        # obtenir le code html de la page
        with open(corpus, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)
        root = ET.Element("corpus")
        corpus = soup.corpus
        all_documents = corpus.find_all("document")
        for doc in all_documents :
            num_article = doc.article.get_text()
            # val_balise = doc.balise.get_text()    # ici comment faire pour qu'il prenne le pbalise du paramettre ?
            val_balise = doc.find(balise).get_text()
            document = ET.SubElement(root, "document")
            article = ET.SubElement(document, "article")
            article.text = num_article
            bal = ET.SubElement(document,balise)
            bal.text = val_balise
        # Écrire dans un fichier
        tree = ET.ElementTree(root)
        tree.write(f"{balise}.xml", encoding="utf-8", xml_declaration=True)
            

    except TypeError as e :
        print("erreur : ", e)



# 1. extraction

In [ ]:
def export(file_name, nb_colonnes, liste_finale):
    if nb_colonnes > 1 :
        with open(file_name, "w") as f:
            for element in liste_finale :
                line = ""
                for i in range(nb_colonnes) :
                    line = line + f"{element[i]} \t" 
                line = line + " \n"
                f.writelines(line)
    if nb_colonnes == 1 :
        with open(file_name, "w") as f:
            for element in liste_finale :
                line = f"{element} \t" 
                f.writelines(line)
    
def extraction_spacy(fichier, balise):
    nlp = spacy.load("fr_core_news_sm")
    try : 
        # obtenir le code html de la page
        with open(fichier, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)

        liste_finale = []
        all_balise = soup.find_all(balise)
        for bal in all_balise : 
            texte = bal.get_text()
            doc = nlp(texte)
            for token in doc :
                cle = [token.text, token.lemma_]
                if cle not in liste_finale :
                    liste_finale.append(cle)
        
        export(DATA/f"ExtracionSpacy{balise}.txt", 2, liste_finale)

    except Exception as e:

        print("erreur  : ", e)
        print(fichier)

def extraction_snowball(fichier, balise):
    try : 
        # obtenir le code html de la page
        with open(fichier, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)

        liste_finale = []
        all_balise = soup.find_all(balise)
        for bal in all_balise : 
            texte = bal.get_text()
            mots = texte.split()

            for mot in mots :
                if mot.isalpha():  # verrifie si le mot n'est constituer que des lettre et pas des chiffres
                    cle = [mot, stemmer.stem(mot.lower())] # mot et sa racine
                    if cle not in liste_finale :
                        liste_finale.append(cle)
        
        export(DATA/f"ExtracionSnowball{balise}.txt", 2, liste_finale)

    except Exception as e:

        print("erreur  : ", e)
        print(fichier)

# partie analyse 

** Nombre de mots uniques

    mesure la taille du vocabulaire

    élevé → vocabulaire riche mais dispersé

    faible → mots bien regroupés mais risque de perte d’infocritère 

Fréquence des mots (distribution)

    mesure combien de fois les mots apparaissent

    bonne distribution → mots bien regroupés

    mauvaise distribution → mots éclatés ou mal normalisés

Taux de réduction du vocabulaire

    mesure combien de mots ont été fusionnés

    élevé → compression forte mais perte de sens possible

    modéré → bon équilibre

In [ ]:
def analyse(fichier, methode):
    with open(fichier) as f :
        lines = f.readlines()
    
    # je sépare le fichier : je met les mots brute dans une 
    # liste et les reduction dans une autres pour faciliter les traitements
    mots = []
    reductions = []
    for line in lines :
        mots.append(line.split()[0])
        reductions.append(line.split()[1])

    # nombre de mot = nombre de mot unique pour les reduction 
    # car il y a potentiellement les doublons 
    nb_mots = len(reductions)
    nb_unique = len(set(reductions))
    print( f"nombre mots pour la methode {methode} : ",nb_mots )
    print( f"nombre mots unique pour la methode {methode} : ",nb_unique )

    # taux  de perte 
    taux = (1 - (nb_unique/nb_mots))*100
    print( f"taux de perte pour la methode {methode} : ", round(taux,2) ) # roud == deux chiffres apres la virgule

    # dsitribution : frequence d'apparition
    dictionnaire = {}
    for mot in reductions :
        if mot not in dictionnaire :
            dictionnaire[mot] = reductions.count(mot)

    print("distribution : -------------")
    for mot, freq in dictionnaire.items():
        print(mot, ":", freq  )
    print("distribution : -------------")

